# Baseline: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_baseline_2026-11-03.md` (B1–B7).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, diffusers.__version__, transformers.__version__)
from diffusers import WanPipeline  # fails here if diffusers is too old

In [ ]:
# Push the repo to GitHub first, or upload the folder and cd into it instead
!git clone https://github.com/FredWen-S/video-dit-l4.git
%cd video-dit-l4

Smoke test: 1 prompt, 17 frames, 4 steps. Checks the download, the pipeline and the script (~5 min including download).

In [ ]:
!python bench/profile_baseline.py --frames 17 --steps 4 --n-prompts 1 --warmup-steps 0 --out results/smoke

Working setting: 33 frames, 50 steps, 4 prompts, with the kernel split (~12 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --steps 50 --kernels --out results/base_33f

Full length: 81 frames, 50 steps, 2 prompts, with the kernel split (~20 min).

In [ ]:
!python bench/profile_baseline.py --frames 81 --steps 50 --n-prompts 2 --kernels --out results/base_81f

Optional: an Nsight Systems timeline of a short run (the phases show up as NVTX ranges). Colab does not ship `nsys`. If the install below fails, skip this cell; the torch.profiler traces above cover the kernel split.

In [ ]:
!apt-get -qq install -y nsight-systems-cli > /dev/null 2>&1 || echo 'nsys install failed, skip'
!which nsys && nsys profile -t cuda,nvtx -o results/nsys_33f_10step --force-overwrite true \
    python bench/profile_baseline.py --frames 33 --steps 10 --n-prompts 1 --no-save-frames --out results/nsys_run

In [ ]:
# Keep results: frames_*.npy are the reference for later quality metrics
!zip -qr baseline_results.zip results && ls -lh baseline_results.zip
from google.colab import files; files.download('baseline_results.zip')